In [1]:
import imageio.v2 as imageio

In [2]:
dog = imageio.imread("../data/p1ch04/image-dog/bobby.jpg")
dog

array([[[ 77,  45,  22],
        [ 77,  45,  22],
        [ 78,  46,  21],
        ...,
        [118,  78,  52],
        [117,  77,  51],
        [116,  76,  50]],

       [[ 75,  43,  20],
        [ 76,  44,  21],
        [ 77,  45,  20],
        ...,
        [118,  78,  52],
        [117,  77,  51],
        [116,  76,  50]],

       [[ 74,  39,  17],
        [ 75,  40,  18],
        [ 77,  43,  18],
        ...,
        [117,  80,  51],
        [117,  78,  49],
        [116,  77,  48]],

       ...,

       [[215, 165,  78],
        [216, 166,  79],
        [217, 167,  80],
        ...,
        [174, 121,  51],
        [176, 123,  53],
        [176, 123,  53]],

       [[215, 165,  78],
        [216, 166,  79],
        [217, 167,  80],
        ...,
        [173, 123,  54],
        [174, 124,  55],
        [174, 124,  55]],

       [[215, 165,  78],
        [216, 166,  79],
        [217, 167,  80],
        ...,
        [159, 109,  40],
        [158, 107,  41],
        [158, 107,  41]]

In [3]:
# At this point, img_arr is a NumPy array-like object with three dimensions: two spatial dimensions, width and height, 
# and a third dimension corresponding to the red, green, and blue channels.
dog.shape

(720, 1280, 3)

In [4]:
# PyTorch modules dealing with image data require tensors to be laid out as C × H × W: channels, height, and width, respectively.

In [5]:
import torch

In [6]:
img_tensor = torch.from_numpy(dog)

In [7]:
out_tensor = img_tensor.permute(2,0,1)
out_tensor

tensor([[[ 77,  77,  78,  ..., 118, 117, 116],
         [ 75,  76,  77,  ..., 118, 117, 116],
         [ 74,  75,  77,  ..., 117, 117, 116],
         ...,
         [215, 216, 217,  ..., 174, 176, 176],
         [215, 216, 217,  ..., 173, 174, 174],
         [215, 216, 217,  ..., 159, 158, 158]],

        [[ 45,  45,  46,  ...,  78,  77,  76],
         [ 43,  44,  45,  ...,  78,  77,  76],
         [ 39,  40,  43,  ...,  80,  78,  77],
         ...,
         [165, 166, 167,  ..., 121, 123, 123],
         [165, 166, 167,  ..., 123, 124, 124],
         [165, 166, 167,  ..., 109, 107, 107]],

        [[ 22,  22,  21,  ...,  52,  51,  50],
         [ 20,  21,  20,  ...,  52,  51,  50],
         [ 17,  18,  18,  ...,  51,  49,  48],
         ...,
         [ 78,  79,  80,  ...,  51,  53,  53],
         [ 78,  79,  80,  ...,  54,  55,  55],
         [ 78,  79,  80,  ...,  40,  41,  41]]], dtype=torch.uint8)

In [8]:
out_tensor.shape

torch.Size([3, 720, 1280])

In [9]:
import os

In [10]:
dir_cats = "../data/p1ch04/image-cats/"
cat_files = [f for f in os.listdir(dir_cats) if f.endswith(".png")]
len(cat_files)

3

In [11]:
# preallocate a tensor with (N, C, H, W) shape that contains batch image data
batch_cats_t = torch.zeros(len(cat_files), 3, 256, 256, dtype=torch.uint8)

In [12]:
# populate the batch tensor
for i, f in enumerate(cat_files):
    img_np = imageio.imread(os.path.join(dir_cats, f))
    img_np_t = torch.from_numpy(img_np)
    img_np_t = img_np_t.permute(2,0,1)
    img_np_t = img_np_t[:3] # exclude alpha channel if the image has it we only need RGB for our purposes
    batch_cats_t[i] = img_np_t
batch_cats_t

tensor([[[[156, 152, 124,  ..., 150, 149, 158],
          [174, 134, 165,  ..., 120, 136, 138],
          [127, 156, 107,  ..., 131, 143, 164],
          ...,
          [116, 130, 129,  ..., 127, 118, 112],
          [129, 130, 123,  ..., 115, 121, 114],
          [129, 123, 118,  ..., 113, 121, 120]],

         [[139, 135, 109,  ..., 135, 135, 147],
          [160, 119, 149,  ..., 105, 122, 124],
          [113, 140,  90,  ..., 118, 129, 152],
          ...,
          [ 99, 110, 111,  ..., 117, 108, 103],
          [111, 111, 106,  ..., 106, 112, 105],
          [111, 104, 102,  ..., 103, 110, 111]],

         [[129, 123,  98,  ..., 131, 132, 145],
          [155, 110, 137,  ..., 102, 119, 121],
          [104, 132,  80,  ..., 112, 125, 146],
          ...,
          [ 93, 108, 105,  ..., 125, 115, 108],
          [108, 108,  98,  ..., 110, 117, 110],
          [107,  98,  95,  ..., 108, 115, 116]]],


        [[[202, 193, 190,  ...,  13,  13,  12],
          [199, 192, 189,  ...,  14

### normalizing the data

In [13]:
# NN best work with input values that belog either [0,1] or [-1, 1]
# One possibility is to divide the values of the pixels by 255 (the maximum representable number in 8-bit unsigned)

In [14]:
batch_cats_t_norm = batch_cats_t.float() # casting
# batch_cats_t_norm /= 256

In [15]:
# Another possibility is to compute the mean and standard deviation of 
# the input data and scale it so that the output has zero mean and unit standard deviation 
# across each channel—a technique commonly known as standardization
n_channels = batch_cats_t_norm.shape[1]
for c in range(n_channels):
    mean_div = torch.mean(batch_cats_t_norm[:, c])
    std_div = torch.std(batch_cats_t_norm[:, c])
    batch_cats_t_norm[:, c] = (batch_cats_t_norm[:, c] - mean_div) / std_div
# batch_cats_t_norm

## 3D images: Volumetric data

In [16]:
# We just have an extra dimension, depth, after the channel dimension, leading to a 5D tensor of shape N × C × D × H × W

In [17]:
ct_dir = "../data/p1ch04/volumetric-dicom/2-LUNG 3.0  B70f-04083/" # CT scan
vol_arr = imageio.volread(ct_dir, "DICOM") # read one volume. Digital Imaging and Communications in Medicine
vol_arr.shape

Reading DICOM (examining files): 99/99 files (100.0%)
  Found 1 correct series.
Reading DICOM (loading data): 99/99  (100.0%)


(99, 512, 512)

In [18]:
vol = torch.from_numpy(vol_arr).float()
vol = torch.unsqueeze(vol,0) # adding channel dimension. unsqueeze adds a new dimension of size 1 at the specified position
vol.shape

torch.Size([1, 99, 512, 512])

## Tabular data

In [19]:
import csv
import numpy as np
wine_path = "../data/p1ch04/tabular-wine/winequality-white.csv"
wineq_np = np.loadtxt(wine_path, dtype=np.float32, delimiter=";", skiprows=1)
wineq_np

array([[ 7.  ,  0.27,  0.36, ...,  0.45,  8.8 ,  6.  ],
       [ 6.3 ,  0.3 ,  0.34, ...,  0.49,  9.5 ,  6.  ],
       [ 8.1 ,  0.28,  0.4 , ...,  0.44, 10.1 ,  6.  ],
       ...,
       [ 6.5 ,  0.24,  0.19, ...,  0.46,  9.4 ,  6.  ],
       [ 5.5 ,  0.29,  0.3 , ...,  0.38, 12.8 ,  7.  ],
       [ 6.  ,  0.21,  0.38, ...,  0.32, 11.8 ,  6.  ]],
      shape=(4898, 12), dtype=float32)

In [20]:
col_list = next(csv.reader(open(wine_path), delimiter=';'))
wineq_np.shape, col_list

((4898, 12),
 ['fixed acidity',
  'volatile acidity',
  'citric acid',
  'residual sugar',
  'chlorides',
  'free sulfur dioxide',
  'total sulfur dioxide',
  'density',
  'pH',
  'sulphates',
  'alcohol',
  'quality'])

In [21]:
wineq_t = torch.from_numpy(wineq_np)
wineq_t.shape, wineq_t.dtype

(torch.Size([4898, 12]), torch.float32)

In [22]:
# performing regression or classification task

In [23]:
data = wineq_t[:, :-1]
data.shape

torch.Size([4898, 11])

In [24]:
target = wineq_t[:,-1]
target.shape

torch.Size([4898])

In [25]:
# here we want to treat the target tensor as a tensor of labels. transforming to the vector of integer scores

In [26]:
target = wineq_t[:, -1].long()
target, target.shape

(tensor([6, 6, 6,  ..., 6, 7, 6]), torch.Size([4898]))

### one-hot encoding

In [27]:
# Что такое формат one-hot?
# Унитарный код (в англоязычной литературе unitary code, one-hot) — двоичный код фиксированной длины, 
# содержащий только одну 1 — прямой унитарный код или только один 0 — обратный (инверсный) унитарный код.

In [28]:
target_onehot = torch.zeros(target.shape[0],10)
target_onehot.shape

torch.Size([4898, 10])

In [29]:
# We can achieve one-hot encoding using the scatter_ method, 
# which fills the tensor with values from a source tensor along the indices provided as arguments
# Let’s see what scatter_ does. First, we notice that its name ends with an underscore. As you learned in the previous chapter,
# this is a PyTorch convention that indicates the method will not return a new tensor, but will instead modify the tensor in place.

In [30]:
target.unsqueeze(1) # A column tensor indicating the indices of the elements to scatter.
# The call to unsqueeze adds a singleton dimension, from a 1D tensor of 4,898 elements to a 2D tensor of size (4,898 × 1),
# without changing its contents. No extra elements are added; we just decided to use an extra index to access the elements. 
# That is, we access the first element of target as target[0] and the first element of its unsqueezed counterpart as target_unsqueezed[0,0].

tensor([[6],
        [6],
        [6],
        ...,
        [6],
        [7],
        [6]])

In [31]:
target_onehot.scatter_(1, target.unsqueeze(1), 1.0)

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [32]:
random_indices = torch.randint(0, len(target),(5,))
random_indices

tensor([1797,  233, 1617,  951, 3387])

In [33]:
print(target[random_indices])

tensor([8, 6, 6, 6, 5])


In [34]:
print(target_onehot[random_indices])

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]])


In [35]:
# obtain the mean and standard deviations for each column
data_mean = torch.mean(data, dim=0)
data_mean

tensor([6.8548e+00, 2.7824e-01, 3.3419e-01, 6.3914e+00, 4.5772e-02, 3.5308e+01,
        1.3836e+02, 9.9403e-01, 3.1883e+00, 4.8985e-01, 1.0514e+01])

In [36]:
data_var = torch.var(data, dim=0)
data_var

tensor([7.1211e-01, 1.0160e-02, 1.4646e-02, 2.5726e+01, 4.7733e-04, 2.8924e+02,
        1.8061e+03, 8.9455e-06, 2.2801e-02, 1.3025e-02, 1.5144e+00])

In [37]:
## normalize the data by subtracting the mean and dividing by the standard deviation, which helps with the learning process

In [38]:
data_norm = data - data_mean / torch.sqrt(data_var)
data_norm

tensor([[ -1.1231,  -2.4905,  -2.4015,  ..., -18.1143,  -3.8422,   0.2561],
        [ -1.8231,  -2.4605,  -2.4215,  ..., -17.8143,  -3.8022,   0.9561],
        [ -0.0231,  -2.4805,  -2.3615,  ..., -17.8543,  -3.8522,   1.5561],
        ...,
        [ -1.6231,  -2.5205,  -2.5715,  ..., -18.1243,  -3.8322,   0.8561],
        [ -2.6231,  -2.4705,  -2.4615,  ..., -17.7743,  -3.9122,   4.2561],
        [ -2.1231,  -2.5505,  -2.3815,  ..., -17.8543,  -3.9722,   3.2561]])

In [39]:
# Finding thresholds

In [40]:
bad_indices = target <= 3

In [41]:
bad_indices.shape, bad_indices.dtype, bad_indices.sum()

(torch.Size([4898]), torch.bool, tensor(20))

In [42]:
bad_data = data[bad_indices] # feature in PyTorch called advanced indexing, we can use a tensor with data type torch.bool to index the data tensor
bad_data.shape

torch.Size([20, 11])

In [43]:
bad_data = data[target <= 3]
mid_data = data[(target > 3) & (target < 7)]
good_data = data[target >=7]

In [44]:
bad_mean = torch.mean(bad_data, dim=0)
mid_mean = torch.mean(mid_data, dim=0)
good_mean = torch.mean(good_data, dim=0)

In [45]:
for i, args in enumerate(zip(col_list, bad_mean, mid_mean, good_mean)):
    print('{:2} {:20} {:6.2f} {:6.2f} {:6.2f}'.format(i, *args))

 0 fixed acidity          7.60   6.89   6.73
 1 volatile acidity       0.33   0.28   0.27
 2 citric acid            0.34   0.34   0.33
 3 residual sugar         6.39   6.71   5.26
 4 chlorides              0.05   0.05   0.04
 5 free sulfur dioxide   53.33  35.42  34.55
 6 total sulfur dioxide 170.60 141.83 125.25
 7 density                0.99   0.99   0.99
 8 pH                     3.19   3.18   3.22
 9 sulphates              0.47   0.49   0.50
10 alcohol               10.34  10.26  11.42


In [46]:
# at first glance, the bad wines seem to have higher total sulfur dioxide, among other differences

In [47]:
total_sulfur_threshold = 141.83
total_sulfur_data = data[:,6]
predicted_indices = torch.lt(total_sulfur_data, total_sulfur_threshold)
predicted_indices.shape, predicted_indices.dtype, predicted_indices.sum()

(torch.Size([4898]), torch.bool, tensor(2727))

In [48]:
# get the indices of the actually good wines

In [49]:
actual_indices = target > 5
actual_indices.shape, actual_indices.dtype, actual_indices.sum()

(torch.Size([4898]), torch.bool, tensor(3258))

In [50]:
n_matches = torch.sum(actual_indices & predicted_indices).item()
n_predicted = torch.sum(predicted_indices).item()
n_actual = torch.sum(actual_indices).item()
n_matches, n_matches / n_predicted, n_matches / n_actual

(2018, 0.74000733406674, 0.6193984039287906)

## time-series

In [51]:
bikes_np = np.loadtxt("../data/p1ch04/bike-sharing-dataset/hour-fixed.csv", delimiter=",", skiprows=1, dtype=np.float32, converters={1:lambda x: float(x[8:10])})
bikes_t = torch.from_numpy(bikes_np) # H (hour) x C (columns)
bikes_t.shape, bikes_t.stride()

(torch.Size([17520, 17]), (17, 1))

In [52]:
# reshaping to 3D tensor D (day) x H x C
# calling view on a tensor returns a new tensor that changes the number of dimensions and the striding information, without changing the storage.
# As a result, we can rearrange our tensor at basically zero cost because no data will be copied
# We use -1 as a placeholder for “however many indexes are left, given the other dimensions and the original number of elements.
daily_bikes = bikes_t.view(-1, 24, bikes_t.shape[1])
daily_bikes.shape, daily_bikes.stride()
# For daily_bikes, the stride is telling us that advancing by 1 along the hour dimension (the second dimension) requires us to advance by 17 places in the storage (or one set of columns). 
# Advancing along the day dimension (the first dimension) requires us to advance by a number of elements equal to the length of a row in the storage times 24 
# (here, 408, which is 17 × 24)

(torch.Size([730, 24, 17]), (408, 17, 1))

In [53]:
# finally we need D x C x H
daily_bikes = daily_bikes.transpose(1,2)
daily_bikes.shape, daily_bikes.stride()

(torch.Size([730, 17, 24]), (408, 1, 17))

In [54]:
first_day = bikes_t[:24].long()

In [55]:
weather_onehot = torch.zeros(first_day.shape[0],4)
weather_onehot

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

In [56]:
first_day[:,9] # weather (1-4 levels)

tensor([1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 2, 2, 2, 2])

In [57]:
weather_onehot.scatter_(dim=1, index=first_day[:,9].unsqueeze(1).long()-1, value=1.0)
weather_onehot

tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.]])

In [58]:
# concat our matrix to original dataset
# the columns of the two datasets are stacked together, or equivalently, the new one-hot-encoded columns are appended to the original dataset.
torch.cat((bikes_t[:24], weather_onehot),1)[:1]

tensor([[ 1.0000,  1.0000,  1.0000,  0.0000,  1.0000,  0.0000,  0.0000,  6.0000,
          0.0000,  1.0000,  0.2400,  0.2879,  0.8100,  0.0000,  3.0000, 13.0000,
         16.0000,  1.0000,  0.0000,  0.0000,  0.0000]])

In [59]:
# do the same for the whole daily_bikes tensor

In [60]:
daily_weather_onehot = torch.zeros(daily_bikes.shape[0], 4, daily_bikes.shape[2])
daily_weather_onehot.shape

torch.Size([730, 4, 24])

In [61]:
daily_weather_onehot.scatter_(1, daily_bikes[:,9,:].unsqueeze(1).long()-1, 1.0)

tensor([[[1., 1., 1.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        ...,

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0

In [62]:
daily_weather_onehot.shape

torch.Size([730, 4, 24])

In [63]:
daily_bikes = torch.cat((daily_bikes, daily_weather_onehot), dim=1)

In [64]:
daily_bikes.shape

torch.Size([730, 21, 24])

In [ ]:
# rescaling variables so they belong to [-1.0,0.0] or [-1.0, 1.0]

In [65]:
temp = daily_bikes[:,10,:]

In [67]:
temp_min = torch.min(temp)

In [68]:
temp_max = torch.max(temp)

In [69]:
temp_min, temp_max

(tensor(0.0200), tensor(1.))

In [70]:
daily_bikes[:,10,:] = ((daily_bikes[:,10,:] - temp_min) / (temp_max - temp_min))

In [71]:
# or subtract the mean and divide by standard deviation

In [73]:
daily_bikes[:,10,:] = ((daily_bikes[:,10,:] - torch.mean(temp)) / torch.std(temp))

### representing text

In [76]:
with open("../data/p1ch04/jane-austen/1342-0.txt", encoding="utf8") as f:
    text = f.read()

In [ ]:
# as the text is a pure English we are going to use ASCI encoding that uses 8 bits

In [77]:
lines = text.split('\n')
line = lines[200]
line

'“Impossible, Mr. Bennet, impossible, when I am not acquainted with him'

In [ ]:
# create a tensor that can hold the total number of one-hot-encoded characters for the whole line

In [79]:
letter_t = torch.zeros(len(line), 128) # 128 hardcoded due to the limits of ASCII. letter_t tensor holds a one-hot-encoded character per row.
letter_t.shape

torch.Size([70, 128])

In [ ]:
# Now, we need to set a 1 on each row in the correct position so that each row represents the correct character. 
# The index where the 1 has to be set corresponds to the index of the character in the encoding

In [83]:
for i, letter in enumerate(line.lower().strip()):
    letter_index = ord(letter) if ord(letter) < 128 else 0
    letter_t[i][letter_index] = 1

In [84]:
letter_t

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

### One-hot encoding whole words

In [92]:
def clean_words(txt):
    puct = '.,;:"!?”“_-'
    words_list = txt.lower().replace("\n"," ").split()
    words_list = [word.strip(puct) for word in words_list]
    return words_list
words_in_line = clean_words(line)
line, words_in_line

('“Impossible, Mr. Bennet, impossible, when I am not acquainted with him',
 ['impossible',
  'mr',
  'bennet',
  'impossible',
  'when',
  'i',
  'am',
  'not',
  'acquainted',
  'with',
  'him'])

In [96]:
word_list = sorted(set(clean_words(text)))

In [97]:
word2index_dict = {word: i for (i, word) in enumerate(word_list)}
len(word2index_dict), word2index_dict['impossible']

(7261, 3394)

In [98]:
word_t = torch.zeros(len(words_in_line), len(word2index_dict))
for i, word in enumerate(words_in_line):
    word_index = word2index_dict[word]
    word_t[i][word_index] = 1
    print('{:2} {:4} {}'.format(i,word_index,word))
word_t.shape

 0 3394 impossible
 1 4305 mr
 2  813 bennet
 3 3394 impossible
 4 7078 when
 5 3315 i
 6  415 am
 7 4436 not
 8  239 acquainted
 9 7148 with
10 3215 him


torch.Size([11, 7261])